# The Trade Finance Doc Mismatch Detector, in one notebook

This is a **working miniature** of `backend/Detector/`. Everything the real package does — classify
each document, extract typed fields from it, cross-check them against each other, issue a verdict —
happens here, in one file, small enough to read in a sitting.

**It runs with no API key.** A fake model stands in for Claude so you can watch the whole machine turn
over offline. The last section shows how to point it at the real thing.

### What was simplified

| Real package | This notebook | Why |
|---|---|---|
| 8 document families | **2** (LC, commercial invoice) | Two is the smallest number that can disagree — the third through eighth teach nothing the second didn't |
| ~40 extracted fields per doc | ~9 | Same idea, less scrolling |
| 4 deterministic tools | **2** | The dropped two need a transport document, and there isn't one here |
| Prompts in `Config/prompts.yaml` + a validating registry | A plain `dict` | The registry is just "fail at startup, not at request 1" |
| Logfire spans everywhere | `print()` | |
| Token accounting (`TokenUsage`) | dropped | Bookkeeping, not mechanism |
| `ConcurrencyLimiter` across agents | dropped | Explained in §14.3, not coded |
| Retry / usage-limit budgets | dropped | |

**Nothing about the *shape* was simplified.** The fan-out, the type-dispatched routing, the join, the
degrade-don't-crash error handling, and the rule that overrides the model are all here exactly as the
real code has them. Those are the parts worth understanding.

## 0. The problem, for people who don't do trade finance

Two companies on opposite sides of the world want to trade. The seller won't ship without payment;
the buyer won't pay without goods. Neither trusts the other.

A **letter of credit (LC)** breaks the deadlock. The buyer's bank promises: *present me documents
that match these terms exactly, and I pay you — I don't care what actually happened to the goods.*

That last clause is the whole thing. Banks deal in **documents, not goods**. If the LC expires on
15 March and the documents reach the bank on the 20th, the bank refuses to pay — even though the
goods are on the water and the buyer wants them. The rulebook is **UCP 600**, a set of articles
published by the International Chamber of Commerce, with **ISBP 745** as the practice guide.

So a bank clerk sits with a stack of PDFs and cross-checks every field against every other document:

- Is the invoice amount within the credit amount (plus any tolerance the LC states)?
- Did the documents arrive before the credit expired?
- Does the beneficiary on the LC match the seller on the invoice?
- Does the invoice describe the goods exactly as the credit does?
- Is the LC number quoted correctly on every document?

A mismatch is called a **discrepancy**. Industry estimates put the first-presentation discrepancy
rate somewhere around half of all presentations. Each one means delay, a fee, and a re-presentation.

**This project automates the clerk.** In, a pile of document text. Out, a list of discrepancies with
severities, the UCP article each one rests on, and a verdict: `clean`, `needs_review`, or `blocked`.

## 1. The shape of a run

```mermaid
stateDiagram-v2
  ingest
  state fan_out_documents <<fork>>
  classify
  state route_by_document_type <<choice>>
  extract_letter_of_credit: letter of credit
  extract_commercial_invoice: commercial invoice
  skip_unclassified: unknown
  state collect_extractions <<join>>
  reconcile

  [*] --> ingest
  ingest --> fan_out_documents: per document
  fan_out_documents --> classify
  classify --> route_by_document_type
  route_by_document_type --> extract_letter_of_credit
  route_by_document_type --> extract_commercial_invoice
  route_by_document_type --> skip_unclassified
  extract_letter_of_credit --> collect_extractions
  extract_commercial_invoice --> collect_extractions
  skip_unclassified --> collect_extractions
  collect_extractions --> reconcile: all documents
  reconcile --> [*]
```

Read it as: **validate the case once, then one independent pipeline per document running
concurrently, then a single reconciliation over all of them.**

Two model calls happen per document branch: `classify` then `extract`. Plus one `reconcile` at the
end over everything. For a 2-document case that's 2 + 2 + 1 = 5 model calls, but only 2 of them are
ever waiting at the same time.

**Why three separate stages instead of one big prompt?**

| Stage | Job | Why it's alone |
|---|---|---|
| `classify` | which family is this? | Cheap, and it picks the schema for the next stage |
| `extract` | pull typed fields | One schema per family; the model fills a form, it doesn't reason |
| `reconcile` | find discrepancies | Reasons over *clean structured data*, never raw OCR noise |

Handing the reconciler raw text would make it do extraction and judgement at once, and get both
slightly wrong. Splitting them means the hard reasoning step reads a tidy table.

## The cast of types — read this once, then relax

This notebook defines 17 classes (plus three small enums), which sounds like a lot. **You only need
to hold five of them in your head.** The rest is plumbing you can skim — and three of the seventeen
are one-line tags with empty bodies.

### The five that matter

A document changes shape three times as it travels through the pipeline:

```
  RawDocument   ──classify──▶   <Family>Doc   ──extract──▶   ExtractedDocument
  the text in                   just a tag                   typed fields, or an error
                                                                      │
                                                                 reconcile
                                                                      ▼
                                                          ReconciliationReport
                                                          = a verdict + a list of Mismatch
```

| Type | Its one job |
|---|---|
| `RawDocument` | the text of one uploaded document — the only evidence the agents ever see |
| `LetterOfCredit`, `CommercialInvoice` | the typed fields pulled out of one document |
| `ExtractedDocument` | one document after its branch finished: carries **either** a payload **or** an error |
| `Mismatch` | one discrepancy — this is the actual product of the whole system |
| `ReconciliationReport` | the verdict, plus every `Mismatch` found |

### Everything else is plumbing

| Type | Why it exists |
|---|---|
| `CaseInput` | a bag of `RawDocument`s plus the presentation date |
| `Classification` | which family, how sure, and why |
| `RoutedDocument` + 3 empty tag subclasses | lets the router dispatch on **type** instead of a string — see §5 |
| `CaseResult` | what one whole run returns |
| `ToolVerdict` | what the deterministic checks return |
| `DetectorDeps`, `CaseState` | the graph's injected services, and its scratchpad |
| `ExtractionBase` | two lines of shared Pydantic config |

### Why types at all, instead of dicts?

Because **four of them are agent output types** — `Classification`, the two payloads, and
`ReconciliationReport`. That is not bookkeeping. Pydantic AI takes each one, generates a JSON
schema from it, makes the model conform to that schema, validates the reply, and retries with the
error message if validation fails.

So `result.output` hands you a checked `LetterOfCredit` object — not a string you have to parse and
hope about. **Those four classes are the contract with the model.** The others are ordinary Python
plumbing, and you can read them in about a minute.

## 2. Setup

Run the cells top to bottom. You need `pydantic-ai`, `pydantic-graph` and `pydantic` — already in
this project's `.venv`.

If Jupyter can't see the project venv as a kernel, from `backend/`:

```bash
uv pip install --python .venv ipykernel
.venv/bin/python -m ipykernel install --user --name tfdd --display-name "Trade Finance Detector"
```

then pick **Trade Finance Detector** as the kernel.

In [ ]:
from __future__ import annotations

import asyncio
import os
import time
from contextlib import ExitStack
from dataclasses import dataclass, field
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from enum import StrEnum
from typing import Annotated, Any, Literal, cast

from pydantic import BaseModel, ConfigDict, Field, TypeAdapter, ValidationError

print('imports ok')

## 3. Enumerations

`StrEnum`, not `Enum`. That means `document_type == 'letter_of_credit'` is `True`, JSON
serialisation gives you the plain string, and the value drops into a database column with no
converter in between.

`UNKNOWN` is a **real member, not an error case.** A document the classifier can't place still flows
through the pipeline and still appears in the final report. An examiner needs to know that something
was presented which the system couldn't read — silently dropping it is the dangerous option.

In [ ]:
class DocumentType(StrEnum):
    """The document families this miniature understands (the real one has 8)."""

    LETTER_OF_CREDIT = 'letter_of_credit'
    COMMERCIAL_INVOICE = 'commercial_invoice'
    UNKNOWN = 'unknown'


class Severity(StrEnum):
    """How badly a discrepancy hurts the presentation."""

    CRITICAL = 'critical'
    """A documentary discrepancy under LC rules: the bank would refuse."""

    WARNING = 'warning'
    """An ambiguity a human examiner should look at."""

    INFO = 'info'
    """A formatting difference or informational observation."""


class CaseStatus(StrEnum):
    """The overall verdict for a presentation."""

    CLEAN = 'clean'
    NEEDS_REVIEW = 'needs_review'
    BLOCKED = 'blocked'


print(DocumentType.LETTER_OF_CREDIT == 'letter_of_credit')  # True — that's the point of StrEnum

## 4. Extraction payloads — the schema *is* the prompt

One model per document family. This is the single most important idea in the project, so it's worth
slowing down.

```python
model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)
```

Two settings, both load-bearing:

- **`extra='forbid'`** puts `additionalProperties: false` in the generated JSON schema. That's what
  lets the provider enforce the schema strictly instead of accepting invented fields.
- **`use_attribute_docstrings=True`** means the docstring under each field becomes that field's
  `description` in the schema — **and the model reads it at inference time.** So in the real project
  those field docstrings are written as *instructions to the model*, not as notes to the next
  developer.

> ⚠️ **A notebook gotcha, and it bites silently.** `use_attribute_docstrings` works by reading your
> **source file** with `inspect.getsource`. Code typed into a notebook cell has no source file, so
> the feature finds nothing and **quietly emits a schema with no descriptions at all** — no error,
> no warning. In a live run that means the model gets a bare field list and extracts worse.
>
> So the models below spell the important ones out with `Field(description=...)`, which produces the
> identical schema. In `Detector/models/extractions.py` — a real `.py` file — the plain docstring
> form works, and that is what the real project uses. Worth remembering the next time you prototype
> a Pydantic AI schema in a notebook and the extractions come back mysteriously worse.

Every field is `| None = None`. Deliberately. The prompt says "leave it null rather than guessing",
and a missing field is evidence in its own right; a hallucinated one is a compliance failure.

`Decimal` for money, `date` for dates — never `float`, never `str`. Tolerance arithmetic on a float
is how you refuse a compliant presentation for being $0.000001 over.

In [ ]:
class ExtractionBase(BaseModel):
    """Shared configuration for every extraction payload."""

    # extra='forbid' -> additionalProperties: false in the schema, so the provider can
    # enforce it strictly. use_attribute_docstrings is kept to mirror the real project;
    # it needs a source file, so in this notebook it is inert (see the note above).
    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)


class LetterOfCredit(ExtractionBase):
    """Structured fields of a documentary credit (MT700 style)."""

    kind: Literal[DocumentType.LETTER_OF_CREDIT] = DocumentType.LETTER_OF_CREDIT

    lc_number: str | None = None
    issuing_bank: str | None = None
    applicant: str | None = Field(
        default=None, description='The buyer, who asked the bank to issue the credit.'
    )
    beneficiary: str | None = Field(
        default=None,
        description='The seller, who gets paid against a compliant presentation.',
    )
    credit_amount: Decimal | None = None
    currency: str | None = None
    expiry_date: date | None = Field(
        default=None,
        description='Last date a compliant presentation may be made at the stated counters.',
    )
    description_of_goods: str | None = None
    tolerance_percent: Decimal | None = Field(
        default=None,
        description="Amount tolerance stated on the credit, e.g. 5 for 'about' / '+/- 5 pct'.",
    )


class CommercialInvoice(ExtractionBase):
    """Structured fields of a commercial invoice."""

    kind: Literal[DocumentType.COMMERCIAL_INVOICE] = DocumentType.COMMERCIAL_INVOICE

    invoice_number: str | None = None
    invoice_date: date | None = None
    lc_reference_number: str | None = Field(
        default=None,
        description='The LC number quoted on the invoice, which must match the credit.',
    )
    seller: str | None = None
    buyer: str | None = None
    currency: str | None = None
    total_amount: Decimal | None = None
    description_of_goods: str | None = None


print('2 payload models defined')

### `ExtractionPayload`, step by step

This is the one bit of type machinery in the project that isn't obvious on sight, so here it is
slowly — problem first, answer last.

#### Step 1 — the problem

When a document finishes its branch it becomes an `ExtractedDocument`, which has to carry the fields
that were pulled out of it. So what type is that field?

```python
class ExtractedDocument(BaseModel):
    document_id: str
    payload: ???        # <- LetterOfCredit? CommercialInvoice?
```

You can't answer until the run is happening, because the families share almost no fields:

| `LetterOfCredit` | `CommercialInvoice` |
|---|---|
| `lc_number` | `invoice_number` |
| `credit_amount` | `total_amount` |
| `expiry_date` | `invoice_date` |

#### Step 2 — the obvious answer, and why it isn't enough

Python's way to say "one of several types" is a union:

```python
payload: LetterOfCredit | CommercialInvoice
```

While the object is in memory, that's fine. It breaks the moment you save it and read it back —
because **JSON has no classes.** A stored payload is just a bag of keys, and Pydantic has to work out
which class to rebuild. Every field here is optional, so both classes fit almost anything.

Watch it guess, and guess wrong:

In [ ]:
# Two classes with identical field names — exactly the situation JSON leaves you in.
class LooksLikeACredit(BaseModel):
    model_config = ConfigDict(extra='forbid')
    reference: str | None = None
    amount: Decimal | None = None


class LooksLikeAnInvoice(BaseModel):
    model_config = ConfigDict(extra='forbid')
    reference: str | None = None
    amount: Decimal | None = None


untagged = TypeAdapter(LooksLikeACredit | LooksLikeAnInvoice)

row = {'reference': 'INV-4471', 'amount': '268400.00'}  # this row came off an invoice
rebuilt = untagged.validate_python(row)

print('in  :', row)
print('out :', type(rebuilt).__name__, ' <- wrong class, and nothing raised')

No exception, no warning — an invoice came back as a credit. Every downstream check that reads
`payload.credit_amount` would now be reading an invoice total. That is a silent wrong answer, which
is the worst kind.

#### Step 3 — the fix: write the answer into the data

Give every payload class a field whose only job is to say what it is:

```python
class CommercialInvoice(ExtractionBase):
    kind: Literal[DocumentType.COMMERCIAL_INVOICE] = DocumentType.COMMERCIAL_INVOICE
```

Read that line as: ***`kind` is allowed exactly one value, and it is already filled in.***

- `Literal[...]` with a single value ⇒ no other value is legal.
- `= DocumentType.COMMERCIAL_INVOICE` ⇒ you never set it by hand, and **the model never gets to pick
  it** — it isn't a choice the model is offered.

So `kind` rides along into the JSON, and on the way back it says which class to rebuild. A field used
that way is called a **discriminator**: the thing that tells the alternatives apart.

#### Step 4 — `ExtractionPayload` is those two facts, given a name

```python
ExtractionPayload = Annotated[
    LetterOfCredit | CommercialInvoice,   # (1) the choices
    Field(discriminator='kind'),          # (2) how to choose
]
```

In one English sentence: **"one of these two — read `kind` to know which."**

`Annotated[X, Y]` does *not* create a new type. It is `X` with a note `Y` stapled to it, which tools
that care can read. To a type checker `ExtractionPayload` is simply the two-way union; to Pydantic it
is a **tagged union**, and Step 2's guessing is gone.

In [ ]:
ExtractionPayload = Annotated[
    LetterOfCredit | CommercialInvoice,
    Field(discriminator='kind'),
]
"""Any extraction payload, tagged by document type so it round-trips through JSON."""

EXTRACTION_PAYLOAD_TYPES: dict[DocumentType, type[ExtractionBase]] = {
    DocumentType.LETTER_OF_CREDIT: LetterOfCredit,
    DocumentType.COMMERCIAL_INVOICE: CommercialInvoice,
}
"""Maps a classified document type to the payload its extractor must return."""

print(list(EXTRACTION_PAYLOAD_TYPES))

Two names, opposite directions — worth not mixing up:

| | Direction | Used when |
|---|---|---|
| `ExtractionPayload` | `kind` string ➜ class | rebuilding a payload **out of** JSON |
| `EXTRACTION_PAYLOAD_TYPES` | `DocumentType` ➜ class | picking which agent and schema to run **before** extraction |

#### Step 5 — in → out

`TypeAdapter` lets you use `ExtractionPayload` on its own, with no model wrapped around it. Four
things happen below:

| | Input | Output |
|---|---|---|
| **A** | a dict tagged `kind: 'commercial_invoice'` | a **`CommercialInvoice`** — with `'268400.00'` now a `Decimal` and `'2026-03-18'` a `date` |
| **B** | that object ➜ JSON ➜ back | the **same class**, an equal object |
| **C** | a dict with a `kind` we don't have | refused, and the legal tags are listed |
| **D** | a dict with **no** `kind` | refused — no tag, no guess |

In [ ]:
payload_adapter = TypeAdapter(ExtractionPayload)

print('A ── one dict in, the right class out')
row = {
    'kind': 'commercial_invoice',
    'invoice_number': 'INV-4471',
    'total_amount': '268400.00',
    'invoice_date': '2026-03-18',
}
payload = payload_adapter.validate_python(row)
print('    in  :', row)
print('    out :', type(payload).__name__)
print('          total_amount =', repr(payload.total_amount), '<- Decimal, not a string')
print('          invoice_date =', repr(payload.invoice_date), '<- a real date')

print('\nB ── object -> JSON -> object, unchanged')
blob = payload_adapter.dump_json(payload).decode()
print('    json:', blob)
revived = payload_adapter.validate_json(blob)
print('    back:', type(revived).__name__, '| equal to the original?', revived == payload)

print('\nC ── a tag that is not one of ours')
try:
    payload_adapter.validate_python({'kind': 'bill_of_lading', 'invoice_number': 'X'})
except ValidationError as exc:
    print('   ', exc.errors()[0]['msg'])

print('\nD ── no tag at all')
try:
    payload_adapter.validate_python({'invoice_number': 'INV-4471'})
except ValidationError as exc:
    print('   ', exc.errors()[0]['msg'])

#### Step 6 — where it is actually used

Exactly one place:

```python
class ExtractedDocument(BaseModel):
    payload: ExtractionPayload | None = None   # <- the field from Step 1, now typed
```

Note where it is **not** used: the extraction agents in §9 each get one *concrete* class as their
`output_type`, never the union — by the time an agent runs, routing has already decided the family.

So the union isn't there to help the model. It is there for the payload's life **after** extraction:
sitting inside `ExtractedDocument`, serialised into the API response and (in the real project) a
database row, then coming back a `CommercialInvoice` rather than a `dict`.

#### Step 7 — the whole thing in one line

> **`ExtractionPayload` = "one of the N payload classes, and `kind` says which."**
> `kind` is fixed by the class, not chosen by the model, so it is always right.

### What the model actually sees

This is the JSON schema Pydantic AI generates and hands to Claude. Two things to notice:
`"additionalProperties": false` (that's `extra='forbid'`, and it's what lets the provider enforce
the schema strictly), and the descriptions riding along on the fields that carry instruction.

`lc_number` deliberately has none — its name says everything the model needs.

In [ ]:
schema = LetterOfCredit.model_json_schema()

print('additionalProperties:', schema.get('additionalProperties'))
print('required tag       :', schema['properties']['kind'])
print()
for name, prop in schema['properties'].items():
    desc = prop.get('description', '')
    print(f'  {name:24} {desc}' if desc else f'  {name:24} —')

## 5. Documents in flight

A document changes shape three times as it moves through the graph:

```
RawDocument  ──classify──▶  <Family>Doc  ──extract──▶  ExtractedDocument
 (what you                  (a routing                 (what reconciliation
  hand in)                   envelope)                  reads)
```

`RawDocument.text` is the **only** evidence the agents ever see. This package does no OCR — turning
a PDF into text is your job, upstream.

There is deliberately **no field for saying what a document is.** The caller uploads a file; that's
the whole contract. `classify` works out the family from the text, because that is the only version
that survives contact with reality — ask an uploader to label their own upload and you get it
mislabelled or left blank, and either way you can't act on it.

`filename` rides along for reference only. It is never evidence: calling a file `credit.pdf` doesn't
make it a credit.

In [ ]:
class RawDocument(BaseModel):
    """One uploaded document after text extraction, before the model sees it."""

    document_id: str
    """Stable identifier assigned by the caller (upload id, row id, ...)."""

    text: str
    """Plain text pulled out of the PDF/image. The only evidence the agents get."""

    filename: str | None = None
    """Carried for reference only. Never used as evidence of what the document is."""


class CaseInput(BaseModel):
    """The unit of work: every document presented under one credit."""

    case_id: str
    documents: list[RawDocument] = Field(default_factory=list)
    presented_on: date | None = None
    """When the documents reached the bank. Needed for the UCP 600 Art 14(c) check."""


class Classification(BaseModel):
    """What the classifier agent decided about a single document."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    document_type: DocumentType = Field(
        description="The family this document belongs to, or 'unknown' if it doesn't clearly "
        'match one.'
    )
    confidence: float = Field(
        ge=0.0, le=1.0, description='How sure the classifier is, from 0 to 1.'
    )
    reasoning: str = Field(
        description='The markers in the text that drove the decision, in one or two sentences.'
    )


print('input models defined')

### The empty-subclass trick, step by step

The design choice that looks pointless until you see what it buys. Same shape as
`ExtractionPayload` above: the goal is to turn a runtime mistake into a mistake your **type checker**
catches.

#### Step 1 — the version almost every codebase would write

`classify` decided this is a letter of credit. Something has to send it to the LC extractor:

```python
if doc.classification.document_type == DocumentType.LETTER_OF_CREDIT:
    return extract_lc
elif doc.classification.document_type == DocumentType.COMMERCIAL_INVOICE:
    return extract_invoice
elif ...
```

This works, and has one flaw: **nothing checks the chain is complete.** Add a third family, forget
one `elif`, and nothing objects — not the editor, not the type checker. You find out when a real
document falls off the end of the chain in production.

Nothing *can* check it, either. `document_type` is a **value**, and values are compared while the
program runs — a type checker has no way to see which ones you covered.

#### Step 2 — make the family a *type* instead of a value

So `classify` doesn't return a `RoutedDocument` carrying a string. It returns **one of three
subclasses that add no data at all**:

```python
class LetterOfCreditDoc(RoutedDocument): ...    # <- `...` is the entire class body
```

Three classes, three empty bodies. Their only job is to be three *different types* — and then to be
named together as one union, `RoutedDocuments`.

#### Step 3 — dispatch on the type

```python
.branch(builder.match(LetterOfCreditDoc).to(extract_letter_of_credit))
```

`builder.match(SomeClass)` is an `isinstance` test, not a string comparison. Same runtime behaviour
as the `if` chain — but now each branch names a *type*.

#### Step 4 — which is what makes the difference

`Decision` accumulates the types it has handled in a type parameter. So declaring the return as
`Decision[..., RoutedDocuments]` is an **assertion that the branch table covers the whole union**.
Miss a family and the accumulated type is narrower than `RoutedDocuments`, the annotation stops
holding, and the checker says so before you run anything.

| | `if` on a string | `match` on a type |
|---|---|---|
| you forget a family | silent fall-through | the annotation fails |
| when you find out | in production | in your editor |

That guarantee is the entire payoff for three otherwise-pointless classes.

#### Step 5 — why frozen dataclasses, not Pydantic models

They live for the few milliseconds between `classify` and `extract` and never cross a process
boundary, so they need no validation and no JSON schema. `frozen=True` makes them immutable,
`slots=True` drops the per-instance dict — the cheapest object that can carry a type.

In [ ]:
@dataclass(frozen=True, slots=True)
class RoutedDocument:
    """A classified document on its way to an extractor."""

    raw: RawDocument
    classification: Classification


# Three tags. They add no fields — `...` is the entire body. Their only job is to be
# three *different types*, so the router can dispatch with isinstance instead of on a
# string. They inherit __init__, frozen and slots from RoutedDocument.
class LetterOfCreditDoc(RoutedDocument): ...
class CommercialInvoiceDoc(RoutedDocument): ...
class UnclassifiedDoc(RoutedDocument): ...


type RoutedDocuments = LetterOfCreditDoc | CommercialInvoiceDoc | UnclassifiedDoc
"""Every branch the routing decision must handle."""

ROUTED_DOCUMENT_TYPES: dict[DocumentType, type[RoutedDocument]] = {
    DocumentType.LETTER_OF_CREDIT: LetterOfCreditDoc,
    DocumentType.COMMERCIAL_INVOICE: CommercialInvoiceDoc,
    DocumentType.UNKNOWN: UnclassifiedDoc,
}
"""Maps a classifier verdict onto the envelope that routes it."""

print('3 routing envelopes:', [c.__name__ for c in ROUTED_DOCUMENT_TYPES.values()])

### The output of one branch

Six fields. Read them as one sentence: **which document this is, what the classifier thought of it,
and then either its fields or the reason there are none.**

| Field | Comes from | Present when |
|---|---|---|
| `document_id` | the caller, at upload | always |
| `document_type` | the classifier | always |
| `confidence` | the classifier | always |
| `classification_reasoning` | the classifier | always |
| `payload` | the extractor | only if extraction succeeded |
| `error` | whatever went wrong | only if it didn't |

**`payload` and `error` are complementary — exactly one is set.** `is_usable` is just
`payload is not None` given a name, because "did this document produce any evidence?" gets asked in
four different places.

A document whose extraction failed **still arrives at reconciliation**, carrying its `error`. That
turns the failure into a warning in the report instead of a stack trace in your logs. §14.2 shows it
happening.

#### "Doesn't `document_type` duplicate the payload's `kind`?"

It looks like it. It doesn't — and the reason is *when each one is readable*.

**`document_type` is always there. `kind` is only there when there's a payload.**

That is the whole answer, and the interesting case is the one where they don't coexist:

```python
ExtractedDocument(
    document_id='doc-inv',
    document_type=DocumentType.COMMERCIAL_INVOICE,   # ← the classifier still knew
    confidence=0.96,
    classification_reasoning='Has an invoice number, seller/buyer and a total.',
    payload=None,                                    # ← the extractor died
    error='extraction failed: upstream provider error',
)
```

The examiner has to be told *"an invoice was presented and could not be read."* Not *"something was
presented."* If the family lived only inside the payload, then every document without a payload —
exactly the ones that need explaining — would have no family at all.

The mirror case is why `kind` exists: a payload often travels **without** its `ExtractedDocument`.
Stored in its own JSON column, passed to a function that took only the payload, sent as one field of
an API response. Nothing there can consult `document_type`, so the payload carries its own tag.

| | Lives on | Answers | Survives |
|---|---|---|---|
| `document_type` | the envelope | "what was presented?" | a failed or skipped extraction |
| `kind` | the payload | "which class do I rebuild?" | the payload being sent somewhere alone |

And they can't drift apart: routing picks the extractor **from** `document_type`, and that
extractor's `output_type` is the one class whose `kind` is fixed to the matching value. Nobody types
either of them by hand.

In [ ]:
class ExtractedDocument(BaseModel):
    """The output of one document's classify-then-extract branch."""

    document_id: str
    document_type: DocumentType
    """Set by the classifier, so it survives a failed extraction — unlike `payload.kind`."""

    confidence: float
    classification_reasoning: str

    payload: ExtractionPayload | None = None
    """The extracted fields. `None` when the document was unclassifiable or extraction failed."""

    error: str | None = None
    """Why `payload` is `None`, when it is."""

    @property
    def is_usable(self) -> bool:
        """Whether this document can contribute evidence to reconciliation."""
        return self.payload is not None


# The case the two tags are for: no payload, so no `kind` — but still a known family.
unreadable = ExtractedDocument(
    document_id='doc-inv',
    document_type=DocumentType.COMMERCIAL_INVOICE,
    confidence=0.96,
    classification_reasoning='Has an invoice number, seller/buyer and a total.',
    error='extraction failed: upstream provider error',
)
print(f'{unreadable.document_id}: {unreadable.document_type.value}, '
      f'usable={unreadable.is_usable} — the report can still name what came in')

## 6. The findings

**In plain words.** This is the thing the whole system exists to produce. Everything before it is
machinery for filling in one `Mismatch`: *what disagrees, who disagrees about it, how bad it is, and
what to do next.*

`Mismatch` is the unit of output, and its field list is designed around what a bank ops person
actually needs:

- **`code`** — a stable slug (`late_shipment`) so you can aggregate across thousands of cases:
  *"what do we get refused for most often?"*
- **`explanation`** — prose for the examiner reading one case.
- **`observations`** — the conflicting values themselves, *which document said what*. This is what
  lets a UI show the disagreement side by side instead of making the user re-open both PDFs.
  (The real project uses a small `FieldObservation` model here; plain strings do the same job with
  one less class to hold in your head.)
- **`rule_reference`** — the UCP 600 article. A finding a clerk can't trace to a rule is a finding
  they can't defend to the beneficiary.

In [ ]:
class Mismatch(BaseModel):
    """A single discrepancy between two or more presented documents."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    code: str = Field(
        description="Short stable slug for this kind of discrepancy, e.g. 'late_shipment'."
    )
    severity: Severity = Field(
        description='critical = the bank would refuse; warning = a human should look; '
        'info = cosmetic.'
    )
    field: str = Field(
        description="The business field in dispute, e.g. 'Beneficiary' or 'Port of Discharge'."
    )
    explanation: str = Field(
        description='Plain language a bank ops person would understand: what disagrees and '
        'why it matters.'
    )
    documents_involved: list[DocumentType] = Field(
        default_factory=list,
        description='Every document family that takes part in this discrepancy.',
    )
    observations: list[str] = Field(
        default_factory=list,
        description='The conflicting values, one line per document, written as '
        "'document_type.field = value'.",
    )
    rule_reference: str | None = Field(
        default=None,
        description='The UCP 600 article or ISBP 745 paragraph relied on, when one applies.',
    )
    suggested_action: str | None = Field(
        default=None, description='What the beneficiary or the bank would do to cure it.'
    )


class ReconciliationReport(BaseModel):
    """The reconciliation agent's verdict over a full presentation."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    status: CaseStatus = Field(
        description='blocked if any critical finding, needs_review if only warnings, else clean.'
    )
    summary: str = Field(
        description='Two or three sentences an examiner can read before opening the detail.'
    )
    mismatches: list[Mismatch] = Field(
        default_factory=list, description='Every discrepancy found, most severe first.'
    )
    matched_fields: list[str] = Field(
        default_factory=list,
        description='Fields that were cross-checked and agreed across documents.',
    )

    @property
    def critical_count(self) -> int:
        return sum(1 for m in self.mismatches if m.severity is Severity.CRITICAL)

    @property
    def warning_count(self) -> int:
        return sum(1 for m in self.mismatches if m.severity is Severity.WARNING)


class CaseResult(BaseModel):
    """Everything one run produced. This is what an API layer returns."""

    case_id: str
    status: CaseStatus
    report: ReconciliationReport
    documents: list[ExtractedDocument] = Field(default_factory=list)


print('finding models defined')

## 7. Deterministic tools — the arithmetic the model is not allowed to do

**In plain words.** The model is good at *noticing* that an invoice amount and a credit amount ought
to be compared. It is not reliable at doing the comparison. So the noticing stays with the model, and
the arithmetic moves into Python:

| The model decides | Python computes |
|---|---|
| *these two amounts should be compared, and the credit states 5%* | `250000 × 1.05 = 262500`; `268400 − 262500 = 5900` → over |
| *this presentation date should be checked against that expiry* | `2026-03-20` vs `2026-03-15` → 5 days late |

Each tool hands back a ready-made sentence, and the prompt tells the model to quote the figures it
returns — so every number in the final report was computed, not written.

Date and money maths is exactly where language models are **subtly and confidently wrong**, and
exactly where being wrong costs the beneficiary a refusal. So it isn't left to the model.

These are plain Python functions handed to the reconciliation agent as tools. Pydantic AI derives
each tool's JSON schema from the signature and its description from the docstring — the `Args:`
section becomes the per-parameter descriptions. So **these docstrings are written for the model**,
which is why they cite UCP articles.

Two tools here; the real project has four. The two left out — the 21-day presentation period of
UCP 600 Art 14(c) and the insurance cover check of Art 28(f)(ii) — both need documents this
miniature doesn't carry. `Detector/services/tools.py` has them, and they are the same shape: take
the raw values, do the arithmetic in Python, hand back a sentence.

Each returns a small typed verdict carrying the computed numbers, so the finding can quote them.

In [ ]:
_ZERO = Decimal(0)
_HUNDRED = Decimal(100)


class ToolVerdict(BaseModel):
    """What every deterministic check returns. One class, used by both."""

    model_config = ConfigDict(extra='forbid')

    passed: bool = Field(description='True when the check is satisfied.')
    detail: str = Field(
        description='One line stating the computed numbers, for the agent to quote '
        'in its finding.'
    )


def check_amount_tolerance(
    credit_amount: Decimal,
    presented_amount: Decimal,
    tolerance_percent: Decimal = _ZERO,
) -> ToolVerdict:
    """Check a presented amount against a credit amount plus its stated tolerance.

    Use for invoice-against-LC and draft-against-LC amount checks. UCP 600 Art 30(a)
    allows a tolerance only when the credit states one ('about', '+/- 5 pct'); with no
    stated tolerance pass 0 and the credit amount is a hard ceiling.

    Args:
        credit_amount: The amount available under the credit.
        presented_amount: The amount actually drawn or invoiced.
        tolerance_percent: Tolerance the credit permits, as a percentage (5 means 5%).
    """
    permitted = credit_amount * (_HUNDRED + tolerance_percent) / _HUNDRED
    difference = presented_amount - permitted
    passed = difference <= _ZERO
    detail = (
        f'presented {presented_amount} against credit {credit_amount} '
        f'with {tolerance_percent}% tolerance (ceiling {permitted}): '
        f'{"within" if passed else f"over by {difference}"}'
    )
    return ToolVerdict(passed=passed, detail=detail)


def check_date_order(
    earlier_label: str, earlier: date, later_label: str, later: date
) -> ToolVerdict:
    """Check that one date falls on or before another, and report the gap in days.

    Use for the presentation date against the credit's expiry date (UCP 600 Art 6(e)),
    the invoice date against the expiry date, and any other ordering the credit imposes.

    Args:
        earlier_label: Name of the date that must come first, e.g. 'presentation'.
        earlier: The date that must come first.
        later_label: Name of the date that must come second, e.g. 'credit expiry'.
        later: The date that must come second.
    """
    days = (later - earlier).days
    passed = days >= 0
    detail = (
        f'{earlier_label} {earlier.isoformat()} vs {later_label} {later.isoformat()}: '
        f'{"within by" if passed else "late by"} {abs(days)} day(s)'
    )
    return ToolVerdict(passed=passed, detail=detail)


RECONCILIATION_TOOLS = [check_amount_tolerance, check_date_order]
print(f'{len(RECONCILIATION_TOOLS)} deterministic tools registered')

### Try them — no model involved

These are pure functions. Same inputs, same verdict, forever. Unit-test them directly.

In [ ]:
# One dollar over a 5% tolerance is still a discrepancy. This is why money is Decimal.
print(check_amount_tolerance(Decimal('250000'), Decimal('262501'), Decimal(5)).detail)
print(check_amount_tolerance(Decimal('250000'), Decimal('262500'), Decimal(5)).detail)
print()

# A credit expiring 2026-03-15, presented five days late and then five days early.
print(check_date_order('presentation', date(2026, 3, 20), 'credit expiry', date(2026, 3, 15)).detail)
print(check_date_order('presentation', date(2026, 3, 10), 'credit expiry', date(2026, 3, 15)).detail)

## 8. Prompts

In the real project these live in `Config/prompts.yaml`, loaded by a registry that validates the
whole file at startup — missing key, blank value, non-string value — so a typo in the YAML is a
**startup crash naming the offending key**, not a mysteriously bad extraction three hours into
production. They're in YAML so a compliance reviewer can reword a check without a code deploy.

Here they're a dict, because that machinery is plumbing, not mechanism.

Note how little the extractor prompts say. They don't list the fields — **the Pydantic schema
already did that**, via `use_attribute_docstrings`. The prompt only has to set the policy: *only what
is in the text, null rather than a guess.*

In [ ]:
PROMPTS: dict[str, str] = {
    'classifier': """
You classify raw text extracted from an uploaded trade finance document into exactly one of:
letter_of_credit, commercial_invoice, or unknown.

Base the decision only on the text given. Look for characteristic markers:
  - letter_of_credit: 'Applicant'/'Beneficiary', an LC number, an issuing bank
  - commercial_invoice: 'Invoice Number', seller/buyer, line-item pricing, a total amount

If it doesn't clearly match one of these, classify as unknown rather than guessing.
""".strip(),
    'letter_of_credit': """
You extract structured information from a Letter of Credit (LC) document.
Only use information explicitly found in the text provided. If a field is not present,
leave it null rather than guessing. Normalise dates to ISO format (YYYY-MM-DD).
""".strip(),
    'commercial_invoice': """
You extract structured information from a Commercial Invoice document.
Only use information explicitly found in the text provided. If a field is not present,
leave it null rather than guessing. Normalise dates to ISO format (YYYY-MM-DD).
""".strip(),
    'reconciliation': """
You are a trade finance documentary-compliance checker, modeled on how a bank operations
analyst checks documents against UCP 600 rules and ISBP 745 international standards.
You are given structured extractions from trade documents, which you can rely on as the
single source of truth for field values.

Cross-check these fields across all the documents where applicable:
  - Party names: Applicant vs Buyer; Beneficiary vs Seller. Addresses need not be stated
    identically as long as they are within the same country (UCP 600 Art 14(j))
  - Amount and currency: the invoice amount must not exceed the LC amount, subject to any
    tolerance the credit states (UCP 600 Art 30)
  - Goods description: the invoice must strictly correspond with the LC description
    (UCP 600 Art 18(c))
  - Dates: the documents must be presented on or before the credit's expiry date
    (UCP 600 Art 6(d)(i) and Art 6(e))
  - Document references: the LC number quoted on the invoice

For every mismatch, cite exactly which documents disagree and explain the discrepancy in
plain language a bank ops person would understand. Classify the severity:
  - 'critical': anything causing a documentary discrepancy under LC rules (expired credit,
    over-drawn amount, mismatched beneficiary, goods description not corresponding)
  - 'warning': an ambiguity worth a human examiner's look (minor party name variations)
  - 'info': a cosmetic or formatting difference, or an observation that is explicitly
    permitted by the rules and therefore not a discrepancy

Use the deterministic tools for every date and amount comparison rather than computing them
yourself, and quote the figures they return in your findings.
""".strip(),
}

print(f'{len(PROMPTS)} prompts loaded')

## 9. The agents

**In plain words.** An agent here is not a loop or a planner. It is one question, asked once, whose
answer must fit a shape you specified in advance. Four of them: one asks *what is this?*, two ask
*fill in this form*, one asks *what disagrees?*

A Pydantic AI `Agent` is **a model + a system prompt + an output type**. The output type is the part
that matters:

```python
Agent(model, output_type=Classification, instructions=..., ...)
```

`output_type=Classification` means `result.output` is a **validated `Classification` instance** — not
a string you have to parse, not a dict you have to trust. Pydantic AI generates the JSON schema, asks
the provider to conform to it, validates the response, and on a validation failure feeds the error
back to the model and retries.

**Four agents in three tiers**: one classifier, two extractors (one per family), one reconciler.

Why two separate extractors rather than one agent with a dynamic output type? Each family needs its
own system prompt, and a per-family agent gives per-family observability spans and per-family retry
budgets.

**`thinking`** is Pydantic AI's *provider-neutral* reasoning control — pass `'low'`/`'medium'`/
`'high'` and it translates to whatever the provider takes (on Claude, adaptive thinking at that
effort level). Because it's neutral, pointing one stage at a different provider stays valid. The
per-stage defaults encode a judgement about where reasoning is worth paying for:

| Stage | Thinking | Why |
|---|---|---|
| classify | `low` | It's pattern matching |
| extract | `medium` | Messy OCR text, unlabelled fields |
| reconcile | `high` | This is the compliance judgement — the part worth paying for |

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings

HAS_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
MODEL = 'anthropic:claude-opus-5' if HAS_KEY else 'test'

print(f'ANTHROPIC_API_KEY set: {HAS_KEY}  ->  building agents on model: {MODEL!r}')
print('(with no key we build on the stub model; §13 swaps in a scripted fake anyway)')


def build_agents(model: str) -> tuple[Agent, dict[DocumentType, Agent], Agent]:
    """Construct the four agents. Built once, reused; they hold no per-case state."""
    classifier = Agent(
        model,
        name='document_classifier',
        output_type=Classification,
        instructions=PROMPTS['classifier'],
        retries=2,
        model_settings=ModelSettings(max_tokens=2_048, thinking='low'),
    )

    extractors = {
        document_type: Agent(
            model,
            name=f'{document_type.value}_extractor',
            output_type=payload_type,
            instructions=PROMPTS[document_type.value],
            retries=2,
            model_settings=ModelSettings(max_tokens=8_000, thinking='medium'),
        )
        for document_type, payload_type in EXTRACTION_PAYLOAD_TYPES.items()
    }

    reconciler = Agent(
        model,
        name='reconciliation_engine',
        output_type=ReconciliationReport,
        instructions=PROMPTS['reconciliation'],
        tools=RECONCILIATION_TOOLS,
        retries=2,
        model_settings=ModelSettings(max_tokens=16_000, thinking='high'),
    )

    return classifier, extractors, reconciler


classifier_agent, extractor_agents, reconciler_agent = build_agents(MODEL)
print()
print('classifier :', classifier_agent.name)
print('extractors :', [a.name for a in extractor_agents.values()])
print('reconciler :', reconciler_agent.name)

## 10. Deps and state

The graph takes two context objects, and the split between them is the point:

| | What it holds | Mutable? |
|---|---|---|
| `DetectorDeps` | Injected services — the agents, the limits. Same for the whole run | **No** (frozen) |
| `CaseState` | The scratchpad this run accumulates — the audit trail | **Yes** |

`CaseState.record()` appends an event and prints it, which is how you'll watch the run happen. In the
real project these events also stream to a websocket for a live progress feed.

**On thread safety:** the fan-out runs every document's branch as a concurrent task on a single event
loop. `record()` contains no `await`, so it's atomic with respect to the other branches and needs no
lock. That's not luck — it's why the method is shaped that way.

In [ ]:
@dataclass(frozen=True, slots=True)
class DetectorDeps:
    """Immutable dependencies handed to every step in the graph."""

    classifier: Agent
    extractors: dict[DocumentType, Agent]
    reconciler: Agent
    min_classification_confidence: float = 0.5
    """Below this, treat the document as unclassified rather than trusting the label."""

    max_documents_per_case: int = 25
    max_document_chars: int = 120_000


@dataclass(slots=True)
class CaseState:
    """Mutable state for one run of the pipeline."""

    case_id: str
    presented_on: date | None = None
    events: list[str] = field(default_factory=list)
    """Ordered audit trail; safe to stream to the client as it grows."""

    def record(self, stage: str, message: str) -> None:
        """Append an audit event. No `await` inside, so concurrent branches can't interleave it."""
        self.events.append(f'{stage}: {message}')
        print(f'   [{stage:9}] {message}')


print('deps and state defined')

## 11. The graph

**In plain words.** Up to now everything has been definitions — types, prompts, agents, functions.
This section connects them into a route, and the connecting is checked as the code loads, so a wrong
wire is an import error rather than a bad night.

Pydantic Graph builds graphs from **typed step functions**. Four type parameters go into the builder,
and they mean different things:

| Parameter | Meaning |
|---|---|
| `input_type` | What the graph is called with (`CaseInput`) |
| `output_type` | What it returns (`CaseResult`) |
| `deps_type` | Injected services, same all run |
| `state_type` | The scratchpad it accumulates |

The `collect` **join** is declared up front because the fan-out needs to name it. A join waits for
every branch of its parent fork and folds the results with a reducer — here `reduce_list_append`,
Pydantic Graph's built-in "append each result to a list".

`initial_factory=list[ExtractedDocument]` works because calling a generic alias constructs the
underlying type: `list[int]()` is `[]`.

In [ ]:
from pydantic_ai.exceptions import AgentRunError, RunCancelled, UsageLimitExceeded
from pydantic_ai.format_prompt import format_as_xml
from pydantic_graph import (
    Decision,
    Graph,
    GraphBuilder,
    Step,
    StepContext,
    reduce_list_append,
)

COLLECT_ID = 'collect_extractions'
"""Node id of the join. Named so the fan-out can point at it for the empty-case path."""

FAN_OUT_ID = 'fan_out_documents'
"""Node id of the map fork over the case's documents."""

_SEVERITY_ORDER = {Severity.CRITICAL: 0, Severity.WARNING: 1, Severity.INFO: 2}

builder = GraphBuilder(
    name='mini_trade_finance_detector',
    state_type=CaseState,
    deps_type=DetectorDeps,
    input_type=CaseInput,
    output_type=CaseResult,
)

collect = builder.join(
    reduce_list_append,
    initial_factory=list[ExtractedDocument],
    node_id=COLLECT_ID,
)
"""Gathers one `ExtractedDocument` per presented document, in completion order."""

print('builder created')

### Prompt construction

`_document_prompt` wraps one document so the model can see its identity and, importantly, its
**boundaries** — the `<document_text>` tags stop OCR noise from reading as instructions.

`_reconciliation_prompt` renders the extracted documents as XML evidence. Note the `else` branch:
when no presentation date was supplied it explicitly tells the model *not* to raise a
presentation-period finding on a date it had to assume. Without that line, a model asked to check
timeliness will invent a date and find a discrepancy.

In [ ]:
def _document_prompt(raw: RawDocument) -> str:
    """Wrap one document's text so the model sees its identity and its boundaries.

    Nothing here tells the model what the document is. The id and filename are so it
    can refer to the document; the text, fenced off, is the only evidence.
    """
    return (
        f'Document id: {raw.document_id}\n'
        f'Filename: {raw.filename or "unknown"}\n\n'
        f'<document_text>\n{raw.text}\n</document_text>'
    )


def _reconciliation_prompt(state: CaseState, documents: list[ExtractedDocument]) -> str:
    """Render the extracted documents as the evidence for the compliance check."""
    evidence = [
        {
            'document_id': d.document_id,
            'document_type': d.document_type.value,
            'classification_confidence': round(d.confidence, 2),
            'fields': d.payload,
        }
        for d in documents
    ]
    parts = [f'Case: {state.case_id}', f'Documents presented: {len(documents)}']

    if state.presented_on is not None:
        parts.append(f'Date of presentation: {state.presented_on.isoformat()}')
    else:
        parts.append(
            'Date of presentation: not supplied. Do not raise a presentation-period '
            'finding on a date you had to assume.'
        )
    parts.append(
        '\nUse the deterministic tools for every date and amount comparison rather than '
        'computing them yourself, and quote the figures they return in your findings.\n'
    )
    parts.append(format_as_xml(evidence, root_tag='extracted_documents', item_tag='document'))
    return '\n'.join(parts)


print('prompt builders defined')

### Step 1 — `ingest`

A step is an async function taking a `StepContext`, which gives you three things: `ctx.inputs` (this
step's input), `ctx.deps` (injected), `ctx.state` (mutable).

Note what `ingest` **does not** do: truncate. An oversized document raises, because a silently
truncated LC produces a confident, wrong extraction — the worst available failure mode here. Failing
loudly at the door is the right call.

In [ ]:
@builder.step
async def ingest(ctx: StepContext[CaseState, DetectorDeps, CaseInput]) -> list[RawDocument]:
    """Validate the presentation and hand its documents to the fan-out."""
    case = ctx.inputs
    deps = ctx.deps

    if len(case.documents) > deps.max_documents_per_case:
        raise ValueError(
            f'case {case.case_id} has {len(case.documents)} documents, '
            f'above the limit of {deps.max_documents_per_case}'
        )

    for document in case.documents:
        # Reject rather than truncate: a truncated LC extracts confidently and wrongly.
        if len(document.text) > deps.max_document_chars:
            raise ValueError(
                f'document {document.document_id} has {len(document.text)} characters, '
                f'above the limit of {deps.max_document_chars}; split it before submitting'
            )
        if not document.text.strip():
            raise ValueError(f'document {document.document_id} has no extracted text')

    ctx.state.record('ingest', f'accepted {len(case.documents)} document(s)')
    return case.documents


print('step: ingest')

### Step 2 — `classify`

**In plain words.** Ask the model what the document is, then don't take the answer entirely on trust.
Two things can go wrong, and neither is allowed to take down the case.

Two failure modes are handled here, and both **degrade rather than crash**.

**1. The agent call fails.** Look at the exception ordering:

```python
except (UsageLimitExceeded, RunCancelled):
    raise
except AgentRunError as exc:
    ...degrade to UnclassifiedDoc...
```

`UsageLimitExceeded` and `RunCancelled` are *subclasses* of `AgentRunError`, so they're re-raised
**first**. A budget breach or a cancellation should stop the whole case, not be quietly absorbed into
a report. Every other agent failure degrades that one document and lets the others finish.

**2. Low confidence.** A document classified as an invoice at 0.2 confidence is better treated as
unknown than read under the invoice schema — that would produce a plausible-looking extraction of
entirely the wrong fields, which is far more dangerous than an honest "I couldn't read this".

In [ ]:
@builder.step
async def classify(ctx: StepContext[CaseState, DetectorDeps, RawDocument]) -> RoutedDocuments:
    """Decide which document family this text belongs to, and wrap it for routing."""
    raw = ctx.inputs
    deps = ctx.deps

    try:
        result = await deps.classifier.run(_document_prompt(raw))
    except (UsageLimitExceeded, RunCancelled):
        raise  # a budget breach or cancellation must stop the case, not be absorbed
    except AgentRunError as exc:
        ctx.state.record('classify', f'{raw.document_id}: classification failed: {exc}')
        return UnclassifiedDoc(
            raw=raw,
            classification=Classification(
                document_type=DocumentType.UNKNOWN,
                confidence=0.0,
                reasoning=f'classification failed: {exc}',
            ),
        )

    classification = result.output
    threshold = deps.min_classification_confidence

    if classification.confidence < threshold:
        ctx.state.record(
            'classify',
            f'{raw.document_id}: {classification.document_type.value} at '
            f'{classification.confidence:.2f}, below the {threshold:.2f} threshold; '
            'treating as unclassified',
        )
        return UnclassifiedDoc(raw=raw, classification=classification)

    ctx.state.record(
        'classify',
        f'{raw.document_id}: {classification.document_type.value} '
        f'at {classification.confidence:.2f}',
    )
    envelope = ROUTED_DOCUMENT_TYPES[classification.document_type]
    return cast(RoutedDocuments, envelope(raw=raw, classification=classification))


print('step: classify')

### Step 3 — the extractors

The two families differ only in *which agent* and *which payload type*, so the step bodies are
**generated rather than written out twice**. Each still gets its own node id, which is what shows up
in the rendered diagram and in the observability spans.

Same degrade-don't-crash shape as `classify`: a failed extraction returns an `ExtractedDocument` with
`payload=None` and the error text, and that document still reaches reconciliation.

In [ ]:
async def _extract(
    ctx: StepContext[CaseState, DetectorDeps, RoutedDocument],
    document_type: DocumentType,
) -> ExtractedDocument:
    """Run one family's extraction agent over one document."""
    routed = ctx.inputs
    raw = routed.raw

    base = {
        'document_id': raw.document_id,
        'document_type': document_type,
        'confidence': routed.classification.confidence,
        'classification_reasoning': routed.classification.reasoning,
    }

    try:
        result = await ctx.deps.extractors[document_type].run(_document_prompt(raw))
    except (UsageLimitExceeded, RunCancelled):
        raise
    except AgentRunError as exc:
        ctx.state.record('extract', f'{raw.document_id}: extraction failed: {exc}')
        # payload=None + error set: this still reaches reconciliation as a warning.
        return ExtractedDocument(**base, payload=None, error=f'extraction failed: {exc}')

    ctx.state.record('extract', f'{raw.document_id}: extracted {document_type.value}')
    return ExtractedDocument(**base, payload=result.output)


def _extraction_step(
    document_type: DocumentType,
) -> Step[CaseState, DetectorDeps, Any, ExtractedDocument]:
    """Build the graph step that extracts one document family."""

    async def extract(
        ctx: StepContext[CaseState, DetectorDeps, RoutedDocument],
    ) -> ExtractedDocument:
        return await _extract(ctx, document_type)

    return builder.step(
        extract,
        node_id=f'extract_{document_type.value}',
        label=document_type.value.replace('_', ' '),
    )


extract_letter_of_credit = _extraction_step(DocumentType.LETTER_OF_CREDIT)
extract_commercial_invoice = _extraction_step(DocumentType.COMMERCIAL_INVOICE)


@builder.step(node_id='skip_unclassified', label='unknown')
async def skip_unclassified(
    ctx: StepContext[CaseState, DetectorDeps, UnclassifiedDoc],
) -> ExtractedDocument:
    """Carry an unrecognised document through to the join without extracting it.

    It still reaches reconciliation so the examiner sees that something was presented
    which the pipeline could not read.
    """
    routed = ctx.inputs
    ctx.state.record('skip', f'{routed.raw.document_id}: not a known document family')
    return ExtractedDocument(
        document_id=routed.raw.document_id,
        document_type=DocumentType.UNKNOWN,
        confidence=routed.classification.confidence,
        classification_reasoning=routed.classification.reasoning,
        payload=None,
        error='document type could not be determined; not extracted',
    )


print('steps: 2 extractors + skip_unclassified')

### The rule that overrides the model

**In plain words.** The model is asked for a verdict, and then the code ignores its answer and works
the verdict out itself — because "is this blocked?" is a lookup, not an opinion:

| If the findings contain… | Verdict |
|---|---|
| any `critical` | `blocked` |
| otherwise, any `warning` | `needs_review` |
| otherwise | `clean` |

This is the difference between a system a bank can use and a demo.

The reconciliation prompt asks the model for a `status`, and `ReconciliationReport` has the field.
But the mapping from findings to verdict is a **rule, not a judgement** — so the rule wins:

```python
return report.model_copy(update={'mismatches': ..., 'status': _derive_status(mismatches)})
```

A report that lists a critical finding and claims `clean` comes back `blocked`. You'll see it happen
in §14.1.

`_finalise_report` also sorts findings most-severe-first, and appends a warning for any document that
failed to extract — so an unreadable scan can't silently shrink the evidence base without the
examiner being told.

In [ ]:
def _derive_status(mismatches: list[Mismatch]) -> CaseStatus:
    """Map findings to a verdict. A rule, not a judgement — so it overrides the model."""
    severities = {m.severity for m in mismatches}
    if Severity.CRITICAL in severities:
        return CaseStatus.BLOCKED
    if Severity.WARNING in severities:
        return CaseStatus.NEEDS_REVIEW
    return CaseStatus.CLEAN


def _finalise_report(
    report: ReconciliationReport, documents: list[ExtractedDocument]
) -> ReconciliationReport:
    """Sort the findings, enforce the status rule, and flag unreadable documents."""
    mismatches = sorted(report.mismatches, key=lambda m: _SEVERITY_ORDER[m.severity])

    unreadable = [d for d in documents if not d.is_usable]
    if unreadable:
        mismatches.append(
            Mismatch(
                code='document_not_extracted',
                severity=Severity.WARNING,
                field='document set',
                explanation=(
                    f'{len(unreadable)} presented document(s) could not be read into structured '
                    'fields and took no part in the cross-checks: '
                    + ', '.join(f'{d.document_id} ({d.error})' for d in unreadable)
                ),
                documents_involved=[DocumentType.UNKNOWN],
                suggested_action='Re-upload a clearer copy, or examine these documents manually.',
            )
        )
        mismatches.sort(key=lambda m: _SEVERITY_ORDER[m.severity])

    # The status the model wrote is discarded here in favour of the derived one.
    return report.model_copy(
        update={'mismatches': mismatches, 'status': _derive_status(mismatches)}
    )


def _no_evidence_report(documents: list[ExtractedDocument]) -> ReconciliationReport:
    """The verdict when nothing could be extracted, so there is nothing to compare."""
    detail = (
        'No documents were presented.'
        if not documents
        else f'None of the {len(documents)} presented document(s) could be read into fields.'
    )
    return ReconciliationReport(
        status=CaseStatus.NEEDS_REVIEW,
        summary=f'{detail} No cross-document checks were performed.',
        mismatches=[
            Mismatch(
                code='no_usable_documents',
                severity=Severity.WARNING,
                field='document set',
                explanation=detail,
                documents_involved=[DocumentType.UNKNOWN],
                suggested_action='Check the uploads and the text extraction step, then resubmit.',
            )
        ],
    )


print('report post-processing defined')

### Step 4 — `reconcile`

**In plain words.** Everything before this was preparation. One call, every document's typed fields
side by side, one question: what disagrees?

The only step that sees every document at once. If nothing could be extracted it skips the model
call entirely — there is nothing to compare, and asking anyway would invite invention.

In [ ]:
@builder.step
async def reconcile(
    ctx: StepContext[CaseState, DetectorDeps, list[ExtractedDocument]],
) -> CaseResult:
    """Cross-check the extracted documents and assemble the case result."""
    state = ctx.state
    documents = sorted(ctx.inputs, key=lambda d: d.document_id)
    usable = [d for d in documents if d.is_usable]

    if not usable:
        report = _no_evidence_report(documents)
        state.record('reconcile', 'no usable extractions; skipped the compliance check')
    else:
        result = await ctx.deps.reconciler.run(_reconciliation_prompt(state, usable))
        report = _finalise_report(result.output, documents)
        state.record(
            'reconcile',
            f'{report.status.value}: {report.critical_count} critical, '
            f'{report.warning_count} warning',
        )

    return CaseResult(
        case_id=state.case_id, status=report.status, report=report, documents=documents
    )


print('step: reconcile')

### Wiring it together

Two things to notice.

**The fan-out.** `.map()` is the fork. `ingest` returns `list[RawDocument]`; `.map()` splits that list
and sends **each element down its own concurrent branch**, so `classify` receives a single
`RawDocument`, not the list.

`downstream_join_id=COLLECT_ID` handles the degenerate case — mapping an *empty* list. Without it, a
case with zero documents would fan out to nothing and the join would wait forever. With it, the fork
jumps straight to the join, which yields its empty initial value, and reconciliation proceeds to
report "no documents presented".

**`build()` validates at import time.** Every node reachable from start, no dead ends, end node
reachable. A miswired graph fails when the module loads — not on the first request in production.

In [ ]:
def _routing_decision() -> Decision[CaseState, DetectorDeps, RoutedDocuments]:
    """The branch table from a classified document to its extractor.

    Branches match on the envelope class, so adding a document family without adding a
    branch here is a type error rather than a silent fall-through.
    """
    return (
        builder.decision(node_id='route_by_document_type', note='UCP 600 document families')
        .branch(builder.match(LetterOfCreditDoc).to(extract_letter_of_credit))
        .branch(builder.match(CommercialInvoiceDoc).to(extract_commercial_invoice))
        .branch(builder.match(UnclassifiedDoc).to(skip_unclassified))
    )


EXTRACTION_STEPS = (
    extract_letter_of_credit,
    extract_commercial_invoice,
    skip_unclassified,
)
"""Everything that can feed the join. Ordering only affects the rendered diagram."""

builder.add(
    builder.edge_from(builder.start_node).to(ingest),
    builder.edge_from(ingest)
    .label('per document')
    .map(fork_id=FAN_OUT_ID, downstream_join_id=COLLECT_ID)  # ← the fan-out
    .to(classify),
    builder.edge_from(classify).to(_routing_decision()),
    builder.edge_from(*EXTRACTION_STEPS).to(collect),
    builder.edge_from(collect).label('all documents').to(reconcile),
    builder.edge_from(reconcile).to(builder.end_node),
)

case_graph: Graph[CaseState, DetectorDeps, CaseInput, CaseResult] = builder.build()
print('graph built and validated')

### The diagram comes from the wiring

Because it's derived from the graph itself, it **cannot drift out of date**. Compare this to the
diagram in §1 — same shape.

In [ ]:
print(case_graph.render())

## 12. A sample case, with discrepancies planted on purpose

Two documents under one credit, presented on 20 March 2026. Read them as a clerk would, and see how
many problems you can spot before the pipeline tells you.

There are **two critical discrepancies and one red herring** in here.

In [ ]:
LC_TEXT = """
IRREVOCABLE DOCUMENTARY CREDIT
LC Number: LC-2026-88431
Issuing Bank: Meridian Commercial Bank, Singapore
Applicant: Harborline Trading Pte Ltd, Singapore
Beneficiary: Anand Textiles Pvt Ltd, Tirupur, India
Amount: USD 250,000.00
Tolerance: +/- 5 PCT
Expiry Date: 2026-03-15 at counters of issuing bank
Description of Goods: 40,000 pcs 100% cotton knitted t-shirts, CIF Singapore
Documents required: signed commercial invoice, packing list.
"""

INVOICE_TEXT = """
COMMERCIAL INVOICE
Invoice No: INV-4471          Date: 2026-03-18
L/C Ref: LC-2026-88431
Seller: Anand Textiles Pvt. Ltd., Plot 14 Mangalam Road, Tirupur 641604, India
Buyer: Harborline Trading Pte Ltd, Singapore
Description: 40,000 pcs 100% cotton knitted t-shirts, CIF Singapore
Total Amount: USD 268,400.00
"""

sample_case = CaseInput(
    case_id='case-001',
    presented_on=date(2026, 3, 20),
    documents=[
        RawDocument(document_id='doc-lc', text=LC_TEXT, filename='credit.pdf'),
        RawDocument(document_id='doc-inv', text=INVOICE_TEXT, filename='invoice.pdf'),
    ],
)

print(f'case {sample_case.case_id}: {len(sample_case.documents)} documents, '
      f'presented {sample_case.presented_on}')

<details>
<summary><b>Click to reveal what's wrong</b> (try to find them first)</summary>

1. **Amount over the credit** — invoice USD 268,400 against a credit of 250,000 with a stated 5%
   tolerance. The ceiling is 262,500, so it's over by 5,900. *Critical.* (UCP 600 Art 18(b), 30(a))
2. **Presented after expiry** — the credit expired 2026-03-15; the documents reached the bank on
   2026-03-20, five days late. Nothing cures this: an expired credit is no longer available.
   *Critical.* (Art 6(d)(i), 6(e))
3. **The red herring** — the invoice gives the seller's full street address, `Plot 14 Mangalam Road,
   Tirupur 641604`, where the credit says only `Tirupur, India`, and abbreviates the legal form as
   `Pvt. Ltd.` rather than `Pvt Ltd`. This is **not** a discrepancy: under Art 14(j) the addresses
   need not be stated identically, so long as they are within the same country — and they are.
   Marked `info`.

The goods description is fine, incidentally: the invoice repeats the credit's wording exactly, which
is what Art 18(c) demands of an invoice specifically. And the invoice date (2026-03-18) is itself
after expiry, which is a symptom of the same problem as finding 2 rather than a separate one.

</details>

## 13. Running it offline

`FunctionModel` lets you script a model's responses. Ours reads the document id out of the prompt and
returns a canned classification, extraction, or report — so the whole graph runs in milliseconds with
no key and no network.

`agent.override(model=...)` swaps the model **for the duration of a context manager** — the agents
themselves are untouched. This is exactly how you'd test the real pipeline (the real project uses
`TestModel` the same way, and `custom_output_args` gets validated against the agent's real
`output_type`, so stubs can't drift away from the schemas).

The scripted reconciler below reports only the findings whose documents actually reached it, so
§14.2 — where one document fails to extract — gets an honest answer rather than a canned one.

In [ ]:
from pydantic_ai.messages import ModelMessage, ModelResponse, ToolCallPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

FAKE_CLASSIFICATIONS = {
    'doc-lc': ('letter_of_credit', 0.97, 'Names an issuing bank, an applicant and a beneficiary.'),
    'doc-inv': ('commercial_invoice', 0.96, 'Has an invoice number, seller/buyer and a total.'),
}

FAKE_EXTRACTIONS = {
    'doc-lc': {
        'kind': 'letter_of_credit',
        'lc_number': 'LC-2026-88431',
        'issuing_bank': 'Meridian Commercial Bank, Singapore',
        'applicant': 'Harborline Trading Pte Ltd, Singapore',
        'beneficiary': 'Anand Textiles Pvt Ltd, Tirupur, India',
        'credit_amount': '250000.00',
        'currency': 'USD',
        'expiry_date': '2026-03-15',
        'description_of_goods': '40,000 pcs 100% cotton knitted t-shirts, CIF Singapore',
        'tolerance_percent': '5',
    },
    'doc-inv': {
        'kind': 'commercial_invoice',
        'invoice_number': 'INV-4471',
        'invoice_date': '2026-03-18',
        'lc_reference_number': 'LC-2026-88431',
        'seller': 'Anand Textiles Pvt. Ltd., Plot 14 Mangalam Road, Tirupur 641604, India',
        'buyer': 'Harborline Trading Pte Ltd, Singapore',
        'currency': 'USD',
        'total_amount': '268400.00',
        'description_of_goods': '40,000 pcs 100% cotton knitted t-shirts, CIF Singapore',
    },
}


def _prompt_text(messages: list[ModelMessage]) -> str:
    """Everything the agent was sent, flattened into one string."""
    return '\n'.join(
        part.content
        for message in messages
        for part in getattr(message, 'parts', [])
        if isinstance(getattr(part, 'content', None), str)
    )


def _document_id_from(messages: list[ModelMessage]) -> str:
    """Pull the document id back out of the prompt, so the fake can answer per document."""
    text = _prompt_text(messages)
    if 'Document id: ' not in text:
        return ''
    return text.split('Document id: ', 1)[1].split('\n', 1)[0].strip()


def _output_tool(info: AgentInfo) -> str:
    """The name of the tool Pydantic AI generated from the agent's output_type."""
    assert info.output_tools, 'this agent has no output tool'
    return info.output_tools[0].name


async def fake_classifier(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    document_type, confidence, reasoning = FAKE_CLASSIFICATIONS[_document_id_from(messages)]
    return ModelResponse(parts=[ToolCallPart(_output_tool(info), {
        'document_type': document_type, 'confidence': confidence, 'reasoning': reasoning,
    })])


async def fake_extractor(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    return ModelResponse(
        parts=[ToolCallPart(_output_tool(info), FAKE_EXTRACTIONS[_document_id_from(messages)])]
    )


print('fake classifier + extractor defined')

In [ ]:
FAKE_SUMMARY = (
    'Two critical discrepancies: the invoice is drawn over the credit even after the 5% '
    'tolerance, and the documents reached the bank five days after the credit expired.'
)

FAKE_MATCHED_FIELDS = [
    'applicant / buyer',
    'beneficiary / seller (name)',
    'LC number quoted on the invoice',
    'currency',
    'description of goods (strictly corresponds, Art 18(c))',
]

FAKE_MISMATCHES = [
    {
        'code': 'amount_over_credit',
        'severity': 'critical',
        'field': 'Amount',
        'explanation': (
            'The invoice is for USD 268,400.00 against a credit of USD 250,000.00 with a '
            'stated 5% tolerance, a ceiling of USD 262,500.00: over by USD 5,900.00.'
        ),
        'documents_involved': ['letter_of_credit', 'commercial_invoice'],
        'observations': [
            'letter_of_credit.credit_amount = USD 250000.00 +/- 5%',
            'commercial_invoice.total_amount = USD 268400.00',
        ],
        'rule_reference': 'UCP 600 Art 18(b), Art 30(a)',
        'suggested_action': 'Amend the credit, or present an invoice within the ceiling.',
    },
    {
        # Only the LC is needed for this one — see §14.2, where the invoice never extracts
        # and this finding still stands.
        'code': 'presented_after_expiry',
        'severity': 'critical',
        'field': 'Presentation date',
        'explanation': (
            'Presentation 2026-03-20 vs credit expiry 2026-03-15: late by 5 day(s). The '
            'credit was no longer available when the documents reached the bank.'
        ),
        'documents_involved': ['letter_of_credit'],
        'observations': [
            'letter_of_credit.expiry_date = 2026-03-15',
            'case.presented_on = 2026-03-20',
        ],
        'rule_reference': 'UCP 600 Art 6(d)(i), Art 6(e)',
        'suggested_action': 'Nothing cures an expired credit; ask the applicant for a re-issue.',
    },
    {
        'code': 'beneficiary_address_variance',
        'severity': 'info',
        'field': 'Beneficiary address',
        'explanation': (
            "The invoice gives the seller as 'Anand Textiles Pvt. Ltd., Plot 14 Mangalam "
            "Road, Tirupur 641604, India' where the credit names 'Anand Textiles Pvt Ltd, "
            "Tirupur, India'. Addresses need not be stated identically provided they are "
            'within the same country, so this is acceptable.'
        ),
        'documents_involved': ['letter_of_credit', 'commercial_invoice'],
        'observations': [
            'letter_of_credit.beneficiary = Anand Textiles Pvt Ltd, Tirupur, India',
            'commercial_invoice.seller = Anand Textiles Pvt. Ltd., Plot 14 Mangalam Road, '
            'Tirupur 641604, India',
        ],
        'rule_reference': 'UCP 600 Art 14(j)',
    },
]


async def fake_reconciler(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    """Answer from whatever evidence actually arrived.

    A fixed canned report would claim an amount discrepancy even in §14.2, where the
    invoice never extracts — so the fake keeps only the findings whose documents are
    present in the prompt. A real reconciler behaves the same way, for the same reason.
    """
    evidence = _prompt_text(messages)
    mismatches = [
        m
        for m in FAKE_MISMATCHES
        if all(document_type in evidence for document_type in m['documents_involved'])
    ]
    complete = len(mismatches) == len(FAKE_MISMATCHES)
    return ModelResponse(parts=[ToolCallPart(_output_tool(info), {
        # Whatever status the fake writes here is discarded by _finalise_report — §14.1.
        'status': 'blocked',
        'summary': FAKE_SUMMARY if complete else (
            'Not every document could be read, so the findings below cover only the '
            'documents that were.'
        ),
        'matched_fields': FAKE_MATCHED_FIELDS if complete else [],
        'mismatches': mismatches,
    })])


async def run_offline(
    case: CaseInput,
    classifier_fn=fake_classifier,
    extractor_fn=fake_extractor,
    reconciler_fn=fake_reconciler,
) -> CaseResult:
    """Run the graph with every agent's model swapped for a scripted fake."""
    deps = DetectorDeps(
        classifier=classifier_agent,
        extractors=extractor_agents,
        reconciler=reconciler_agent,
    )
    with ExitStack() as stack:
        stack.enter_context(classifier_agent.override(model=FunctionModel(classifier_fn)))
        for agent in extractor_agents.values():
            stack.enter_context(agent.override(model=FunctionModel(extractor_fn)))
        stack.enter_context(reconciler_agent.override(model=FunctionModel(reconciler_fn)))

        return await case_graph.run(
            state=CaseState(
                case_id=case.case_id, presented_on=case.presented_on
            ),
            deps=deps,
            inputs=case,
        )


print('fake reconciler + runner defined')

### Run it

Watch the audit trail. The two `classify` lines appear together, then the two `extract` lines —
that's the fan-out. They arrive in **completion order, not document order**, which is why anything
consuming these events must key on `document_id` rather than on arrival position.

In [ ]:
started = time.perf_counter()
result = await run_offline(sample_case)
print()
print(f'{result.status.value.upper()}   ({(time.perf_counter() - started) * 1000:.0f} ms)')

### The report

In [ ]:
def print_report(result: CaseResult) -> None:
    """Render a CaseResult the way an examiner's screen might."""
    report = result.report
    bar = '=' * 78

    print(bar)
    print(f'CASE {result.case_id}   VERDICT: {report.status.value.upper()}')
    print(bar)
    print()
    print('SUMMARY')
    print(f'  {report.summary}')
    print()
    print(f'DOCUMENTS ({len(result.documents)})')
    for d in result.documents:
        mark = 'ok  ' if d.is_usable else 'FAIL'
        print(f'  [{mark}] {d.document_id:9} {d.document_type.value:20} '
              f'confidence {d.confidence:.2f}')
        if d.error:
            print(f'         {d.error}')
    print()
    print(f'FINDINGS ({report.critical_count} critical, {report.warning_count} warning, '
          f'{len(report.mismatches)} total)')
    print()
    for i, m in enumerate(report.mismatches, 1):
        print(f'  {i}. [{m.severity.value.upper()}] {m.field} — {m.code}')
        print(f'     {m.explanation}')
        for obs in m.observations:
            print(f'       · {obs}')
        if m.rule_reference:
            print(f'     rule: {m.rule_reference}')
        if m.suggested_action:
            print(f'     cure: {m.suggested_action}')
        print()
    if report.matched_fields:
        print('CROSS-CHECKED AND AGREED')
        for f in report.matched_fields:
            print(f'  · {f}')
    print(bar)


print_report(result)

### The extracted data itself

Every document came back as a typed, validated object. `Decimal` amounts, real `date` objects — not
strings that look like dates.

In [ ]:
by_id = {d.document_id: d for d in result.documents}

for document in result.documents:
    if document.payload is None:
        continue
    print(f'--- {document.document_id}  ({type(document.payload).__name__}) ---')
    for name, value in document.payload.model_dump(exclude_none=True).items():
        print(f'  {name:24} {value!r}')
    print()

# Proof the types are real, not strings that look like them:
print('doc-inv total_amount type:', type(by_id['doc-inv'].payload.total_amount).__name__)
print('doc-lc  expiry_date  type:', type(by_id['doc-lc'].payload.expiry_date).__name__)

### One document's journey, concretely

The same document, in each of the three shapes from the map at the top.

In [ ]:
raw_lc = sample_case.documents[0]
final_lc = next(d for d in result.documents if d.document_id == 'doc-lc')

print('1. RawDocument       ', f'{raw_lc.document_id}: {len(raw_lc.text)} chars of plain text')
print('2. LetterOfCreditDoc ', 'same data, but a type the router can dispatch on (transient)')
print('3. ExtractedDocument ', f'{final_lc.document_type.value}, usable={final_lc.is_usable}')
print()
print('   .payload is a     ', type(final_lc.payload).__name__)
print('   .credit_amount    ', repr(final_lc.payload.credit_amount))
print('   .expiry_date      ', repr(final_lc.payload.expiry_date))
print('   .error            ', repr(final_lc.error))

## 14. Three things worth seeing

### 14.1 — The rule beats the model

Hand `_finalise_report` a report where the model claims `clean` while listing a critical finding.

In [ ]:
lying_report = ReconciliationReport(
    status=CaseStatus.CLEAN,                       # ← the model's claim
    summary='All documents agree.',
    mismatches=[
        Mismatch(
            code='presented_after_expiry',
            severity=Severity.CRITICAL,            # ← but it also reported this
            field='Presentation date',
            explanation='Presented 5 days after the credit expired.',
        )
    ],
)

print('model said :', lying_report.status.value)
print('rule says  :', _finalise_report(lying_report, []).status.value)
print()
print('A critical finding means blocked. Not negotiable, not the model\'s call.')

### 14.2 — One bad document degrades; the case survives

The invoice's extraction now fails with an HTTP 500. Without the `except AgentRunError` handling,
that would take down the whole case and you'd get a stack trace instead of a report.

Instead the case finishes, and the report tells you exactly what it lost:

- the **LC still extracts**, so the check that needs only the LC — presentation against expiry —
  still runs and is still `critical`;
- the amount check **can't** run, because the amount it needed was on the invoice, so that finding
  is simply absent rather than invented;
- a **warning** names the document that took no part in the cross-checks.

That last one is the point. A shrinking evidence base is itself a finding — otherwise a clean-looking
report could just mean half the documents failed to load.

In [ ]:
from pydantic_ai.exceptions import ModelHTTPError


async def flaky_extractor(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    """Fails for doc-inv only. ModelHTTPError is an AgentRunError, so the step catches it."""
    if _document_id_from(messages) == 'doc-inv':
        raise ModelHTTPError(status_code=500, model_name='test', body='upstream provider error')
    return await fake_extractor(messages, info)


degraded = await run_offline(sample_case, extractor_fn=flaky_extractor)

print()
print(f'the case still finished: {degraded.status.value}')
print()
for d in degraded.documents:
    print(f'  {d.document_id:9} usable={str(d.is_usable):5} {d.error or ""}')
print()
print('what survived, and what the report says about what did not:')
for m in degraded.report.mismatches:
    print(f'  [{m.severity.value:8}] {m.code}')

### 14.3 — The fan-out really is concurrent

Give every model call a 300 ms delay and time the run.

There is also **no phase barrier**: a document's extraction starts as soon as *its own*
classification finishes — it doesn't wait for the other documents. Only the join is a barrier, and it
has to be, because reconciliation needs every document.

In [ ]:
def slow(fn):
    """Wrap a fake model function with 300ms of latency."""
    async def wrapped(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
        await asyncio.sleep(0.3)
        return await fn(messages, info)
    return wrapped


started = time.perf_counter()
await run_offline(
    sample_case,
    classifier_fn=slow(fake_classifier),
    extractor_fn=slow(fake_extractor),
    reconciler_fn=slow(fake_reconciler),
)
elapsed = time.perf_counter() - started

print()
print('2 documents, 5 model calls, 0.30s each')
print(f'  if fully sequential : {5 * 0.3:.2f}s')
print(f'  parallel floor      : {3 * 0.3:.2f}s   (classify | extract | reconcile)')
print(f'  measured            : {elapsed:.2f}s')

### What that means at scale — and the ceiling this notebook leaves out

Unbounded fan-out is the real risk. A 20-document presentation opens 20 simultaneous provider
connections and collects `429`s.

The real project fixes this with a **single `ConcurrencyLimiter` shared by all ten agents**:

```python
limiter = ConcurrencyLimiter(max_running=6, max_queued=None, name='detector-model-calls')
# then passed to every Agent as max_concurrency=limiter
```

One limiter object for all of them — that's the whole point. Every agent bills the same provider
account against the same rate limit, so **the ceiling has to be shared**. Eight extractors with a
limit of six *each* is not a limit of six.

Measured in the real project with 12 documents at 200 ms per call:

| `max_parallel_model_calls` | Peak concurrent calls | Wall time |
|---|---|---|
| `None` | 12 | 0.68s |
| `6` (default) | 6 | 1.07s |
| `2` | 2 | 2.71s |

It's omitted here only to keep the agent-building cell short.

## 15. Running it against the real Claude

Everything above ran on a scripted fake. To use the real thing:

```bash
export ANTHROPIC_API_KEY=sk-ant-...
```

then **restart the kernel and re-run** — `MODEL` in §9 is chosen at build time.

The cell below is a no-op without a key. With one, it runs the whole pipeline for real: 5 model calls
on `claude-opus-5` across three stages. Expect it to take a while; `reconcile` runs at `thinking='high'`
because the compliance judgement is where the reasoning budget earns its cost.

**This costs money.** For a cheaper run, put `classify` and `extract` on `anthropic:claude-sonnet-5`
and keep only `reconcile` on Opus — the real project's config exposes exactly that split
(`DETECTOR_CLASSIFIER_MODEL`, `DETECTOR_EXTRACTION_MODEL`, `DETECTOR_RECONCILIATION_MODEL`), which is
the obvious place to trade cost for accuracy.

In [ ]:
if not HAS_KEY:
    print('No ANTHROPIC_API_KEY — skipping the live run.')
    print('Set it, restart the kernel, re-run from the top, and this cell will do the real thing.')
else:
    real_deps = DetectorDeps(
        classifier=classifier_agent,
        extractors=extractor_agents,
        reconciler=reconciler_agent,
    )
    live = await case_graph.run(
        state=CaseState(
            case_id='case-001-live',
            presented_on=sample_case.presented_on,
        ),
        deps=real_deps,
        inputs=sample_case,
    )
    print()
    print_report(live)

### Cheaper: a per-stage model split

Rebuild the agents with a small model doing the mechanical work and the strong one doing the
judgement. This mirrors what the real project's settings let you do without a code change.

In [ ]:
# Uncomment to try, with a key set. Classification and extraction are form-filling;
# reconciliation is the part that actually reasons.
#
# cheap_classifier, cheap_extractors, _ = build_agents('anthropic:claude-sonnet-5')
# _, _, strong_reconciler = build_agents('anthropic:claude-opus-5')
#
# split_deps = DetectorDeps(
#     classifier=cheap_classifier,
#     extractors=cheap_extractors,
#     reconciler=strong_reconciler,
# )
# split = await case_graph.run(
#     state=CaseState(case_id='case-001-split', presented_on=sample_case.presented_on),
#     deps=split_deps,
#     inputs=sample_case,
# )
# print_report(split)

print('(commented out — uncomment with a key set)')

## 16. Back to the real repository

Every section here maps to a file:

| This notebook | `backend/` | What the real one adds |
|---|---|---|
| §3 enums | `Detector/models/enums.py` | 8 document families |
| §4 payloads | `Detector/models/extractions.py` | 8 payloads, ~40 fields each, nested line items |
| §5 documents | `Detector/models/documents.py` | 9 routing envelopes, `TokenUsage` accounting |
| §6 findings | `Detector/models/reconciliation.py` | `FieldObservation` model, `missing_documents`, timing + usage on `CaseResult` |
| §7 tools | `Detector/services/tools.py` | + `check_presentation_period` (Art 14(c)) and `check_insurance_coverage` (Art 28(f)(ii)); richer typed verdicts |
| §8 prompts | `Config/prompts.yaml` + `Detector/prompts/registry.py` | YAML, validated at startup |
| §9 agents | `Detector/services/agents.py` | 10 agents, shared `ConcurrencyLimiter`, Anthropic prompt caching |
| §10 deps/state | `Detector/services/deps.py` | `StageEvent` objects, `RunUsage` accumulation |
| §11 graph | `Detector/services/graph.py` | 9 branches; otherwise identical |
| — | `Detector/services/pipeline.py` | The public facade: `run()` and `run_with_progress()` |
| — | `Detector/core/config.py` | Every tunable as a `DETECTOR_`-prefixed env var |
| — | `Detector/core/observability.py` | Logfire spans across graph, agents and models |

The cross-checks the miniature can't do are the ones that need a transport document — late shipment,
port mismatches, the 21-day presentation period. Adding the bill of lading back is a payload class,
an envelope class, a prompt, a branch, and nothing else. That was the point of the shape.